Create a csv file with the county FIPS code and the following 2017 census data:
- Land area in square kilometers
- Water area in square kilometers
- Farm area in square kilometers
- Crop area in square kilometers

In [11]:
# geopandas not included in erdos environment, need to download
# Download Census TIGER/Line shapefules here: https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html

import geopandas as gpd
import pandas as pd

counties = gpd.read_file('../../work/tl_2017_us_county/tl_2017_us_county.shp')


In [12]:
counties = counties.rename(
    columns = {
        'GEOID': 'FIPS',
    }
)

counties.head()

,STATEFP,COUNTYFP,COUNTYNS,FIPS,NAME,NAMELSAD,LSAD,CLASSFP,MTFCC,CSAFP,CBSAFP,METDIVFP,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
0,31,039,00835841,31039,Cuming,Cuming County,06,H1,G4020,NaN,NaN,NaN,A,1477641638,10701538,+41.9158651,-096.7885168,"POLYGON ((-97.01952 42.0041, -97.01952 42.0049..."
1,53,069,01513275,53069,Wahkiakum,Wahkiakum County,06,H1,G4020,NaN,NaN,NaN,A,680956787,61588406,+46.2946377,-123.4244583,"POLYGON ((-123.43639 46.2382, -123.44759 46.24..."
2,35,011,00933054,35011,De Baca,De Baca County,06,H1,G4020,NaN,NaN,NaN,A,6016761648,29147345,+34.3592729,-104.3686961,"POLYGON ((-104.56739 33.99757, -104.56772 33.9..."
3,31,109,00835876,31109,Lancaster,Lancaster County,06,H1,G4020,339,30700,NaN,A,2169252486,22867561,+40.7835474,-096.6886584,"POLYGON ((-96.9106 40.95841, -96.9106 40.95854..."
4,31,129,00835886,31129,Nuckolls,Nuckolls County,06,H1,G4020,NaN,NaN,NaN,A,1489645186,1718484,+40.1764918,-098.0468422,"POLYGON ((-98.27367 40.0894, -98.27367 40.0894..."


In [13]:
xls = pd.ExcelFile('data/NASSAgcensusDownload2017.xlsx')

crop_df = pd.read_excel(xls, 'Farms', dtype={'FIPSTEXT': str})


In [14]:
desired_columns = ['FIPSTEXT','y17_M059_valueNumeric', 'y17_M063_valueNumeric']

crop_df_acres = crop_df[desired_columns].rename(
    columns = {
        'FIPSTEXT': 'FIPS',
        'y17_M059_valueNumeric': 'farm_acre_as_percent',
        'y17_M063_valueNumeric': 'crop_acre_as_percent'
    }
)

In [15]:
all_data = pd.merge(counties, crop_df_acres, on='FIPS')

all_data.head()

,STATEFP,COUNTYFP,COUNTYNS,FIPS,NAME,NAMELSAD,LSAD,CLASSFP,MTFCC,CSAFP,CBSAFP,METDIVFP,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry,farm_acre_as_percent,crop_acre_as_percent
0,31,039,00835841,31039,Cuming,Cuming County,06,H1,G4020,NaN,NaN,NaN,A,1477641638,10701538,+41.9158651,-096.7885168,"POLYGON ((-97.01952 42.0041, -97.01952 42.0049...",99.5,90.42
1,53,069,01513275,53069,Wahkiakum,Wahkiakum County,06,H1,G4020,NaN,NaN,NaN,A,680956787,61588406,+46.2946377,-123.4244583,"POLYGON ((-123.43639 46.2382, -123.44759 46.24...",8.2,3.06
2,35,011,00933054,35011,De Baca,De Baca County,06,H1,G4020,NaN,NaN,NaN,A,6016761648,29147345,+34.3592729,-104.3686961,"POLYGON ((-104.56739 33.99757, -104.56772 33.9...",79.5,0.68
3,31,109,00835876,31109,Lancaster,Lancaster County,06,H1,G4020,339,30700,NaN,A,2169252486,22867561,+40.7835474,-096.6886584,"POLYGON ((-96.9106 40.95841, -96.9106 40.95854...",78.9,67.71
4,31,129,00835886,31129,Nuckolls,Nuckolls County,06,H1,G4020,NaN,NaN,NaN,A,1489645186,1718484,+40.1764918,-098.0468422,"POLYGON ((-98.27367 40.0894, -98.27367 40.0894...",97.1,67.56


In [16]:
county_area = all_data[["FIPS", "ALAND", 'AWATER', 'farm_acre_as_percent', 'crop_acre_as_percent']].copy()

county_area["area_land_km2"] = (
    county_area["ALAND"] / 1_000_000
)

county_area["area_water_km2"] = (
    county_area["AWATER"] / 1_000_000
)

county_area['farm_area_km2'] = (
    county_area['area_land_km2'] * county_area['farm_acre_as_percent'] / 100
)

county_area['crop_area_km2'] = (
    county_area['area_land_km2'] * county_area['crop_acre_as_percent'] / 100
)




In [17]:
county_area.shape

(3078, 9)

In [18]:
county_area.to_csv("../data/county_data_2017.csv", index=False)